<a href="https://colab.research.google.com/github/jaqueuchoab/dados-divas/blob/main/enade_compilado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Algoritmo para junção dos arquivos da base do ENADE 2023**

Uma vez que a base do ENADE é fragmentada em 32 arquivos distintos, este
algoritmo foi desenvolvido com auxílio da *Inteligência Artificial Claude
Sonnet 4.6* para realizar a junção dos arquivos de forma que seja possível
relacionar os dados a nível de curso e instituição.

É de suma importância destacar que, apesar de cada linha do ENADE
corresponder a um aluno, **não é possível identificar os estudantes na base**.
Nesse contexto, as linhas não definem o perfil individual de um aluno e a
base não deve ser utilizada para esse fim, pois as colunas de cada arquivo
foram unidas horizontalmente com base no `CO_CURSO`, sem a perspectiva de
combinação de informações entre arquivos distintos.

**Importações de bibliotecas necessárias:**
- `pandas` — manipulação e compilação dos dados
- `glob` — busca automática dos arquivos na pasta
- `os` — criação automática da pasta de saída
- `re` — extração do número do nome de cada arquivo para ordenação numéricaimport pandas as pd

In [ ]:
import pandas as pd
import glob
import os
import re

**Configurações**

Essa seção permite limitar os arquivos e colunas que irão ser unidos no arquivos final.

Caso nenhum deva ser excluido, basta deixar as constantes vazias e a limitação para teste como 0.



In [ ]:
# arquivos a ignorar
ARQUIVOS_IGNORAR = [
    'microdados2023_arq3.txt'
]

# colunas a ignorar
COLUNAS_IGNORAR = [
    # 'NU_ANO'
]

# TESTE: limitar quantidade de arquivos (0 = todos)
LIMITE_ARQUIVOS = 0

**Passo 1: Carregamento do arquivo 1**

O arquivo 1 é o único que contém o código da instituição (`CO_IES`) junto
ao código do curso (`CO_CURSO`). Ele serve como base inicial da compilação
e como chave de identificação institucional para todos os demais arquivos.

O tratamento do BOM (`ï»¿`) remove um caractere invisível que aparece no
nome da primeira coluna quando o CSV é salvo com encoding `utf-8-sig`.

In [ ]:
arq1 = pd.read_csv('/content/drive/MyDrive/microdados_enade/microdados2023_arq1.txt', sep=';', encoding='latin-1')
arq1.columns = arq1.columns.str.replace('ï»¿', '', regex=False).str.strip()
arq1 = arq1.sort_values('CO_CURSO').reset_index(drop=True)

# remover colunas ignoradas do arquivo 1
colunas_remover_arq1 = [c for c in COLUNAS_IGNORAR if c in arq1.columns]
if colunas_remover_arq1:
    arq1 = arq1.drop(columns=colunas_remover_arq1)
    print(f"Colunas removidas do arquivo 1: {colunas_remover_arq1}")

print(f"Arquivo 1: {len(arq1)} linhas, {len(arq1.columns)} colunas")
print(f"Colunas do arquivo 1: {arq1.columns.tolist()}")

# chave para validação
chave = arq1[['CO_IES', 'CO_CURSO']].drop_duplicates()
print(f"\nCursos únicos:       {chave['CO_CURSO'].nunique()}")
print(f"Instituições únicas: {chave['CO_IES'].nunique()}")
print(f"Nulos em CO_IES:     {chave['CO_IES'].isna().sum()}")

Arquivo 1: 406294 linhas, 10 colunas
Colunas do arquivo 1: ['NU_ANO', 'CO_CURSO', 'CO_IES', 'CO_CATEGAD', 'CO_ORGACAD', 'CO_GRUPO', 'CO_MODALIDADE', 'CO_MUNIC_CURSO', 'CO_UF_CURSO', 'CO_REGIAO_CURSO']

Cursos únicos:       9812
Instituições únicas: 1347
Nulos em CO_IES:     0


**Passo 2: Listagem e ordenação dos arquivos**

Os arquivos restantes são listados e ordenados **numericamente** — não
alfabeticamente — para garantir o carregamento sequencial.

O arquivo 1 é excluído da lista pois já foi carregado separadamente.
Se `LIMITE_ARQUIVOS` for maior que zero, apenas os primeiros N arquivos
serão processados (útil para testes).

In [ ]:
arquivos = glob.glob('/content/drive/MyDrive/microdados_enade/microdados2023_arq*.txt')
arquivos = [a for a in arquivos if 'microdados2023_arq1.txt' not in a]

# ordenar pelo número extraído do nome do arquivo
arquivos = sorted(arquivos, key=lambda x: int(re.search(r'(\d+)', os.path.basename(x)).group()))

# aplicar filtro de arquivos ignorados
if ARQUIVOS_IGNORAR:
    arquivos = [a for a in arquivos if os.path.basename(a) not in ARQUIVOS_IGNORAR]
    print(f"\nArquivos ignorados: {ARQUIVOS_IGNORAR}")

# aplicar limite de teste
if LIMITE_ARQUIVOS > 0:
    arquivos = arquivos[:LIMITE_ARQUIVOS]


print(f"\nArquivos encontrados ({len(arquivos)}):")
for a in arquivos:
    print(f"  {a}")


Arquivos ignorados: ['microdados2023_arq3.txt']

Arquivos encontrados (30):
  /content/drive/MyDrive/microdados_enade/microdados2023_arq2.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq4.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq5.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq6.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq7.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq8.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq9.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq10.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq11.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq12.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq13.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq14.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_arq15.txt
  /content/drive/MyDrive/microdados_enade/microdados2023_ar

**Passo 3: Base inicial**

O arquivo 1 se torna a base inicial da compilação. Todos os demais arquivos
serão adicionados a ele horizontalmente, coluna a coluna.

In [ ]:
base = arq1.copy()

print(f"\nBase inicial (arquivo 1): {len(base)} linhas, {len(base.columns)} colunas")


Base inicial (arquivo 1): 406294 linhas, 10 colunas


**Passo 4: Compilação dos arquivos**

Para cada arquivo da lista:
1. Carrega o arquivo e corrige o encoding das colunas
2. Ordena pelo `CO_CURSO` para garantir alinhamento com a base
3. Remove colunas definidas em `COLUNAS_IGNORAR`
4. Identifica colunas que ainda não existem na base
5. Adiciona apenas as colunas novas horizontalmente com `concat(axis=1)`

Colunas já presentes na base são ignoradas automaticamente para evitar
duplicatas — por exemplo, `NU_ANO` que aparece em vários arquivos.

In [ ]:
for arq in arquivos:
    df = pd.read_csv(arq, sep=';', encoding='latin-1')
    df.columns = df.columns.str.replace('ï»¿', '', regex=False).str.strip()
    df = df.sort_values('CO_CURSO').reset_index(drop=True)

    # remover colunas ignoradas
    colunas_remover = [c for c in COLUNAS_IGNORAR if c in df.columns]
    if colunas_remover:
        df = df.drop(columns=colunas_remover)

    # ignorar colunas que já existem na base
    colunas_novas = [c for c in df.columns if c not in base.columns]

    if not colunas_novas:
        print(f"{arq} → sem colunas novas, ignorado")
        continue

    base = pd.concat([base, df[colunas_novas]], axis=1)

    print(f"{arq} adicionado → {len(base)} linhas, {len(base.columns)} colunas")

/content/drive/MyDrive/microdados_enade/microdados2023_arq2.txt adicionado → 406294 linhas, 13 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq4.txt adicionado → 406294 linhas, 55 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq5.txt adicionado → 406294 linhas, 56 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq6.txt adicionado → 406294 linhas, 57 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq7.txt adicionado → 406294 linhas, 58 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq8.txt adicionado → 406294 linhas, 59 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq9.txt adicionado → 406294 linhas, 60 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq10.txt adicionado → 406294 linhas, 61 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq11.txt adicionado → 406294 linhas, 62 colunas
/content/drive/MyDrive/microdados_enade/microdados2023_arq12.txt adicio

**Passo 5: Validação do resultado**

Verificações básicas para confirmar que a compilação foi bem-sucedida:

- O número de linhas deve permanecer em 406.294
- O número de cursos únicos deve bater com o arquivo 1
- O número de instituições únicas deve bater com o arquivo 1
- Nulos em `CO_IES` devem ser zero

In [ ]:
print(f"\n=== RESULTADO FINAL ===")
print(f"Linhas:              {len(base)}")
print(f"Colunas:             {len(base.columns)}")
print(f"Cursos únicos:       {base['CO_CURSO'].nunique()}")
print(f"Instituições únicas: {base['CO_IES'].nunique()}")
print(f"Nulos em CO_IES:     {base['CO_IES'].isna().sum()}")


=== RESULTADO FINAL ===
Linhas:              406294
Colunas:             83
Cursos únicos:       9812
Instituições únicas: 1347
Nulos em CO_IES:     0


**Passo 6: Exportação**

A base compilada é salva em dois formatos:

- **Parquet** — formato eficiente para uso no pipeline de integração das três bases
- **CSV** — formato aberto para inspeção visual e interoperabilidade

O encoding `utf-8-sig` no CSV garante que acentos apareçam corretamente ao abrir no Excel.


In [ ]:
os.makedirs('data/interim', exist_ok=True)

base.to_parquet('data/interim/enade_compilado.parquet', index=False)
base.to_csv('data/interim/enade_compilado.csv', sep=';',
            encoding='utf-8-sig', index=False)

print(f"\nBase salva em data/interim/")


Base salva em data/interim/
